# DrawNet — Backbone Comparison

Train DrawNet with different pretrained backbones and compare their performance.

**Backbones compared:**
- `resnet50` — 25.6M params, 2048-dim features
- `efficientnet_b3` — 12.2M params, 1536-dim features  
- `convnext_tiny` — 28.6M params, 768-dim features

**How to use:**
1. Runtime → Change runtime type → T4 GPU
2. Run cells 1–6 once (they set up Drive, deps, code, and data)
3. In **cell 7**, set `BACKBONE` to the backbone you want to train
4. Run cells 7–10 to train and save results
5. Repeat steps 3–4 for each backbone
6. Run the final **Comparison** cell to see the full results table

All data is shared across runs (cached in Drive). Only the model weights differ.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/DrawNet'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive mounted. Project persists at: {DRIVE_DIR}')

## 2. Install dependencies

In [ ]:
!pip install -q kaggle scikit-learn tqdm pyyaml pandas pillow matplotlib
import torch
print(f'PyTorch: {torch.__version__}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT AVAILABLE — check Runtime type!"}')

## 3. Set up project code

In [ ]:
import os, shutil

REPO_DIR = '/content/drawnet-sketch-analysis'
WORK_DIR = f'{REPO_DIR}/drawnet'   # actual project root (contains src/, configs/, etc.)

# Always pull fresh code from GitHub (weights are separate in Drive)
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

GITHUB_REPO = 'sibbala123/drawnet-sketch-analysis'
!git clone https://github.com/{GITHUB_REPO} {REPO_DIR}

os.chdir(WORK_DIR)
print(f'Working directory: {os.getcwd()}')
!ls

## 4. Kaggle credentials

In [ ]:
import os, json

KAGGLE_USERNAME = 'saka2453'          # <-- fill in
KAGGLE_KEY      = '76acc1a5eef0c6d27ca3edf7ed8d94b0'  # <-- fill in

os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
kaggle_json = os.path.expanduser('~/.kaggle/kaggle.json')
with open(kaggle_json, 'w') as f:
    json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
os.chmod(kaggle_json, 0o600)
print('Kaggle credentials written.')

## 5. Download / restore datasets from Drive cache

In [ ]:
import shutil

# ── QuickDraw ──────────────────────────────────────────────────────────────
QD_RAW   = 'data/raw/quickdraw'
QD_DRIVE = f'{DRIVE_DIR}/data/raw/quickdraw'
os.makedirs(QD_RAW, exist_ok=True)

if os.path.exists(QD_DRIVE) and len(os.listdir(QD_DRIVE)) >= 10:
    print('QuickDraw found in Drive — copying...')
    shutil.copytree(QD_DRIVE, QD_RAW, dirs_exist_ok=True)
else:
    print('Downloading QuickDraw from GCS...')
    !gsutil -m cp \
      gs://quickdraw_dataset/full/numpy_bitmap/face.npy \
      gs://quickdraw_dataset/full/numpy_bitmap/house.npy \
      gs://quickdraw_dataset/full/numpy_bitmap/tree.npy \
      gs://quickdraw_dataset/full/numpy_bitmap/cat.npy \
      gs://quickdraw_dataset/full/numpy_bitmap/car.npy \
      gs://quickdraw_dataset/full/numpy_bitmap/bird.npy \
      gs://quickdraw_dataset/full/numpy_bitmap/dog.npy \
      gs://quickdraw_dataset/full/numpy_bitmap/fish.npy \
      gs://quickdraw_dataset/full/numpy_bitmap/flower.npy \
      gs://quickdraw_dataset/full/numpy_bitmap/bicycle.npy \
      {QD_RAW}/
    os.makedirs(QD_DRIVE, exist_ok=True)
    shutil.copytree(QD_RAW, QD_DRIVE, dirs_exist_ok=True)
    print('QuickDraw saved to Drive.')

print(f'QuickDraw files: {len(os.listdir(QD_RAW))}')

# ── TU-Berlin ──────────────────────────────────────────────────────────────
TUB_RAW   = 'data/raw/tuberlin'
TUB_DRIVE = f'{DRIVE_DIR}/data/raw/tuberlin'

if os.path.exists(TUB_DRIVE) and len(os.listdir(TUB_DRIVE)) >= 20:
    print('TU-Berlin found in Drive — copying...')
    shutil.copytree(TUB_DRIVE, TUB_RAW, dirs_exist_ok=True)
else:
    print('Downloading TU-Berlin via Kaggle...')
    !python src/download_tuberlin.py --extract
    os.makedirs(TUB_DRIVE, exist_ok=True)
    shutil.copytree(TUB_RAW, TUB_DRIVE, dirs_exist_ok=True)
    print('TU-Berlin saved to Drive.')

print(f'TU-Berlin folders: {len(os.listdir(TUB_RAW))}')

## 6. Restore cache from Drive

Copies the existing augmented cache (images + index.csv) from Drive.
If `train.csv`/`test.csv` are missing, runs only `split_dataset.py` (seconds, not 15 min).

In [ ]:
import shutil, pandas as pd, subprocess

CACHE_DRIVE = f'{DRIVE_DIR}/data/augmented'
CACHE_LOCAL = 'data/augmented'
os.makedirs(CACHE_LOCAL, exist_ok=True)

def copy_images():
    local_imgs = f'{CACHE_LOCAL}/images'
    drive_imgs = f'{CACHE_DRIVE}/images'

    if os.path.exists(local_imgs):
        print(f'  images/ already on local disk.')
        return True

    # Verify Drive images folder is accessible and non-empty
    try:
        n_drive = len(os.listdir(drive_imgs))
    except Exception as e:
        print(f'  WARNING: Cannot read Drive images folder ({e})')
        n_drive = 0

    if n_drive == 0:
        print('  WARNING: images/ not found in Drive — training will fail.')
        print('  You need to re-run cache_dataset.py to rebuild the image cache.')
        return False

    print(f'  Copying {n_drive} images to local SSD...')
    result = subprocess.run(['cp', '-r', drive_imgs, local_imgs], capture_output=True, text=True)
    if result.returncode != 0:
        print(f'  cp failed: {result.stderr.strip()}')
        print('  Falling back to symlink (training will be slower)...')
        os.symlink(drive_imgs, local_imgs)
    print(f'  Done. {len(os.listdir(local_imgs))} images ready.')
    return True

def restore_csvs():
    for f in ['index.csv', 'train.csv', 'test.csv']:
        src = f'{CACHE_DRIVE}/{f}'
        if os.path.exists(src):
            shutil.copy(src, f'{CACHE_LOCAL}/{f}')

# ── Case 1: splits already in Drive ───────────────────────────────────────
if os.path.exists(f'{CACHE_DRIVE}/train.csv') and os.path.exists(f'{CACHE_DRIVE}/test.csv'):
    print('Cache found in Drive — restoring...')
    restore_csvs()
    copy_images()
    n_train = len(pd.read_csv(f'{CACHE_LOCAL}/train.csv'))
    n_test  = len(pd.read_csv(f'{CACHE_LOCAL}/test.csv'))
    print(f'  train.csv: {n_train} rows  |  test.csv: {n_test} rows')

# ── Case 2: index.csv exists but no split yet ──────────────────────────────
elif os.path.exists(f'{CACHE_DRIVE}/index.csv'):
    print('index.csv found — copying cache and running split_dataset.py only...')
    restore_csvs()
    copy_images()
    !python src/split_dataset.py --show_stats
    shutil.copy(f'{CACHE_LOCAL}/train.csv', f'{CACHE_DRIVE}/train.csv')
    shutil.copy(f'{CACHE_LOCAL}/test.csv',  f'{CACHE_DRIVE}/test.csv')
    print('Splits saved to Drive.')

# ── Case 3: nothing cached → full pipeline ────────────────────────────────
else:
    print('No cache found — running full pipeline (~15 min)...')
    !python src/build_annotations.py --show_stats
    !python src/cache_dataset.py
    !python src/split_dataset.py --show_stats
    os.makedirs(CACHE_DRIVE, exist_ok=True)
    subprocess.run(['cp', '-r', f'{CACHE_LOCAL}/images', f'{CACHE_DRIVE}/images'])
    for f in ['index.csv', 'train.csv', 'test.csv']:
        shutil.copy(f'{CACHE_LOCAL}/{f}', f'{CACHE_DRIVE}/{f}')
    print('Cache saved to Drive.')

In [ ]:
import os

local_imgs = f'{CACHE_LOCAL}/images'
n = len(os.listdir(local_imgs)) if os.path.exists(local_imgs) else 0
print(f'Images on local disk: {n}')

if n >= 64000:
    print('OK — all images present, safe to train.')
elif n > 0:
    print(f'WARNING: only {n}/64158 images present — cache may be incomplete.')
else:
    print('ERROR: no images found — re-run cache_dataset.py before training.')

## 7. Select backbone

**Edit this cell** to choose which backbone to train. Then run cells 8–10.

In [ ]:
# ─── EDIT THIS ───────────────────────────────────────────────────────────────
BACKBONE = 'efficientnet_b3'   # choices: 'resnet50' | 'efficientnet_b3' | 'convnext_tiny'
# ─────────────────────────────────────────────────────────────────────────────

CKPT_DRIVE = f'{DRIVE_DIR}/outputs/{BACKBONE}/checkpoints'
CKPT_LOCAL = f'outputs/{BACKBONE}/checkpoints'
os.makedirs(CKPT_LOCAL, exist_ok=True)

resume_flag = ''
if os.path.exists(f'{CKPT_DRIVE}/best.pt'):
    import shutil
    shutil.copy(f'{CKPT_DRIVE}/best.pt', f'{CKPT_LOCAL}/best.pt')
    epoch_ckpts = sorted([f for f in os.listdir(CKPT_DRIVE) if f.startswith('epoch_')])
    if epoch_ckpts:
        latest = epoch_ckpts[-1]
        shutil.copy(f'{CKPT_DRIVE}/{latest}', f'{CKPT_LOCAL}/{latest}')
        resume_flag = f'--resume {CKPT_LOCAL}/{latest}'
        print(f'Resuming {BACKBONE} from {latest}')
    else:
        resume_flag = f'--resume {CKPT_LOCAL}/best.pt'
        print(f'Resuming {BACKBONE} from best.pt')
else:
    print(f'No checkpoint found — training {BACKBONE} from scratch.')

print(f'Backbone : {BACKBONE}')
print(f'Resume   : {resume_flag or "(none)"}')

## 8. Write backbone config

In [ ]:
import yaml

with open('configs/config_colab.yaml') as f:
    cfg = yaml.safe_load(f)

cfg['model']['backbone'] = BACKBONE

run_cfg_path = f'configs/config_{BACKBONE}.yaml'
with open(run_cfg_path, 'w') as f:
    yaml.dump(cfg, f)

print(f'Config written: {run_cfg_path}')
print(f"  backbone  : {cfg['model']['backbone']}")
print(f"  pretrained: {cfg['model']['pretrained']}")
print(f"  epochs    : {cfg['training']['epochs']}")
print(f"  batch_size: {cfg['training']['batch_size']}")

## 9. Train

In [ ]:
!python -u src/train.py \
    --config {run_cfg_path} \
    --run_name {BACKBONE} \
    {resume_flag}

## 10. Back up to Drive

In [ ]:
import shutil

for folder in [f'outputs/{BACKBONE}/checkpoints',
               f'outputs/{BACKBONE}/logs',
               f'outputs/{BACKBONE}/results']:
    src = f'{WORK_DIR}/{folder}'
    dst = f'{DRIVE_DIR}/{folder}'
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f'Backed up {folder}')

print(f'Done. {BACKBONE} outputs saved to Drive.')

## 11. Training curves for this run

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

log_path = f'outputs/{BACKBONE}/logs/train_log.csv'
if os.path.exists(log_path):
    log = pd.read_csv(log_path)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(f'Training curves — {BACKBONE}', fontsize=13)

    axes[0].plot(log['epoch'], log['train_loss'], label='Train')
    axes[0].plot(log['epoch'], log['val_loss'],   label='Val')
    axes[0].set_title('Total Loss'); axes[0].legend()

    axes[1].plot(log['epoch'], log['intent_top1_acc'])
    axes[1].set_title('Intent Top-1 Accuracy')
    axes[1].axhline(0.033, color='gray', linestyle='--', label='random baseline')
    axes[1].legend()

    axes[2].plot(log['epoch'], log['dev_macro_f1'])
    axes[2].set_title('Deviation Macro F1')

    plt.tight_layout()
    os.makedirs(f'outputs/{BACKBONE}/results', exist_ok=True)
    plt.savefig(f'outputs/{BACKBONE}/results/training_curves.png', dpi=150)
    plt.show()
    print(log[['epoch', 'val_loss', 'intent_top1_acc', 'dev_macro_f1']].tail(10).to_string(index=False))
else:
    print(f'No log found at {log_path}')

---
## 12. Backbone Comparison Table

Run this cell after training all backbones. Reads test_metrics.csv from each backbone's Drive folder.

In [ ]:
import os, sys
import pandas as pd
import matplotlib.pyplot as plt
import torch
import time

sys.path.insert(0, 'src')
from model import DrawNet

BACKBONES = ['resnet50', 'efficientnet_b3', 'convnext_tiny']

rows = []
for bb in BACKBONES:
    metrics_path = f'{DRIVE_DIR}/outputs/{bb}/results/test_metrics.csv'
    if not os.path.exists(metrics_path):
        print(f'  {bb}: no results yet (run training first)')
        continue

    m = pd.read_csv(metrics_path).iloc[0]

    # Param count (no weights needed)
    dummy = DrawNet(backbone_name=bb, pretrained=False, freeze_backbone=False)
    total_p, _ = dummy.param_count()
    del dummy

    # Inference speed (random batch on CPU)
    speed_model = DrawNet(backbone_name=bb, pretrained=False, freeze_backbone=False).eval()
    dummy_input = torch.randn(32, 3, 224, 224)
    with torch.no_grad():
        for _ in range(3):   # warmup
            speed_model(dummy_input)
        t0 = time.perf_counter()
        for _ in range(10):
            speed_model(dummy_input)
        ms_per_batch = (time.perf_counter() - t0) / 10 * 1000
    del speed_model

    rows.append({
        'Backbone':         bb,
        'Params (M)':       f'{total_p/1e6:.1f}',
        'ms/batch (CPU)':   f'{ms_per_batch:.0f}',
        'Intent Top-1 (%)': f'{m["intent_top1_acc"]*100:.1f}',
        'Intent Top-5 (%)': f'{m["intent_top5_acc"]*100:.1f}',
        'Intent F1':        f'{m["intent_macro_f1"]:.3f}',
        'Dev Subset Acc (%)': f'{m["dev_subset_acc"]*100:.1f}',
        'Dev Macro F1':     f'{m["dev_macro_f1"]:.3f}',
        'AUROC Rotation':   f'{m["auroc_rotation"]:.3f}',
        'AUROC Closure':    f'{m["auroc_closure_failure"]:.3f}',
        'AUROC Spatial':    f'{m["auroc_spatial_disorganization"]:.3f}',
        'AUROC Size':       f'{m["auroc_size_distortion"]:.3f}',
    })

if rows:
    df = pd.DataFrame(rows).set_index('Backbone')
    print('\n=== BACKBONE COMPARISON ===')
    print(df.T.to_string())

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    bbs   = df.index.tolist()
    top1  = [float(v) for v in df['Intent Top-1 (%)']]
    devf1 = [float(v) * 100 for v in df['Dev Macro F1']]
    colors = ['#4C72B0', '#DD8452', '#55A868'][:len(bbs)]

    axes[0].bar(bbs, top1, color=colors)
    axes[0].set_title('Intent Top-1 Accuracy (%)')
    axes[0].set_ylim(0, 100)
    for i, v in enumerate(top1):
        axes[0].text(i, v + 0.5, f'{v:.1f}', ha='center', fontsize=10)

    axes[1].bar(bbs, devf1, color=colors)
    axes[1].set_title('Deviation Macro F1 (%)')
    axes[1].set_ylim(0, 100)
    for i, v in enumerate(devf1):
        axes[1].text(i, v + 0.5, f'{v:.1f}', ha='center', fontsize=10)

    plt.tight_layout()
    os.makedirs(f'{DRIVE_DIR}/outputs/comparison', exist_ok=True)
    plt.savefig(f'{DRIVE_DIR}/outputs/comparison/backbone_comparison.png', dpi=150)
    plt.show()
    df.to_csv(f'{DRIVE_DIR}/outputs/comparison/backbone_comparison.csv')
    print('\nChart saved to Drive/DrawNet/outputs/comparison/')
else:
    print('No results found. Train at least one backbone first.')